# Backpropagation — Exercises

Companion to the note [Backpropagation](Backpropagation.md).

Practise the chain rule on computation graphs, the MLP backward recursion
$\delta^{(l)} = (W^{(l+1)\top}\delta^{(l+1)})\odot\phi'(z^{(l)})$, $\partial L/\partial W^{(l)} = \delta^{(l)}h^{(l-1)\top}$,
the softmax + cross-entropy shortcut $\delta^{(L)} = \hat p - y$, and vanishing gradients. Everything is implemented
from scratch in NumPy and verified with **finite-difference gradient checks**, the standard way to test backprop code.
**Code convention:** samples are rows, so a layer is `Z = H @ W + b` with `W` of shape `(n_in, n_out)`; then
$\nabla_W L = H^\top\Delta$ where $\Delta$ stacks the $\delta$'s as rows.

**15 exercises** · 🧠 Concept ×3 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import expit, softmax as _sp_softmax, log_softmax as _sp_log_softmax
rng = np.random.default_rng(0)

def _num_grad(f, x, eps=1e-6):
    # reference central-difference gradient of scalar f w.r.t. array x (x is modified in place and restored)
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        g[i] = (fp - fm) / (2 * eps)
    return g

## Part A · Concepts

### Exercise 1 · Why not just use finite differences?
*🧠 Concept · ★☆☆*

A network has $P = 10^7$ parameters and one forward pass costs $F$ FLOPs.

1. Roughly what does computing the full gradient with central finite differences cost? And with backprop?
2. Why do we still implement finite differences at all?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Central differences need two forward passes **per parameter**. The note says backprop costs about 2–3 forward passes in total.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Finite differences: $2P\cdot F = 2\times10^7 F$. Backprop (reverse-mode autodiff): $\approx 2$–$3F$ for **all** gradients at once,
   because a scalar loss lets every intermediate $\partial L/\partial z$ be reused by everything upstream. A speed-up of millions.
2. As a **test**: finite differences are slow but almost impossible to get wrong, so they are the reference for gradient checks on tiny
   networks (Exercises 10–13). Use float64, a small $\varepsilon$ ($\sim10^{-5}$–$10^{-6}$), and avoid kinks such as ReLU at 0.

</details>

### Exercise 2 · Vanishing and exploding gradients
*🧠 Concept · ★★☆*

Unroll the recursion $\delta^{(l)} = \big(W^{(l+1)\top}\delta^{(l+1)}\big)\odot\phi'(z^{(l)})$ from layer $L$ down to layer 1.

1. Why does it give a *product* of $L-1$ matrices, and why does that product tend to vanish or explode?
2. For each mitigation in the note (ReLU, residual connections, normalisation, gradient clipping), say which factor of the product it acts on.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Write $\delta^{(l)} = D^{(l)}W^{(l+1)\top}D^{(l+1)}W^{(l+2)\top}\cdots\delta^{(L)}$ with $D^{(l)} = \mathrm{diag}(\phi'(z^{(l)}))$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\delta^{(1)} = D^{(1)}W^{(2)\top}D^{(2)}\cdots W^{(L)\top}\delta^{(L)}$. Its norm behaves roughly like $\prod_l \lVert W^{(l)}\rVert\,\lVert D^{(l)}\rVert$:
   if the typical factor is $<1$ (e.g. sigmoid, $\phi'\le 0.25$) it shrinks **exponentially** with depth; if $>1$ it explodes.
2. **ReLU**: $\phi' = 1$ on active units, so $D$ does not shrink the signal. **Residual connections** $h^{(l)} = h^{(l-1)} + F(h^{(l-1)})$ make each Jacobian
   $I + \partial F/\partial h$, so there is always an identity path. **Normalisation** (BatchNorm/LayerNorm) and good **init** (He/Xavier) keep
   $\lVert W\rVert\lVert D\rVert\approx1$. **Clipping** does not fix the product; it caps the final gradient norm so an explosion cannot wreck a step ([[Training Tricks]]).

</details>

### Exercise 3 · Memory and gradient checkpointing
*🧠 Concept · ★★☆*

1. Which quantities from the forward pass must be stored for the backward pass of $z^{(l)} = W^{(l)}h^{(l-1)} + b^{(l)}$, $h^{(l)} = \phi(z^{(l)})$?
2. Training uses far more memory than inference. Why?
3. Gradient checkpointing stores only every $k$-th activation. What is the trade-off, and what choice of $k$ makes memory $O(\sqrt L)$?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Look at the formulas: $\partial L/\partial W^{(l)}$ needs $h^{(l-1)}$, and $\phi'(z^{(l)})$ needs $z^{(l)}$ (or $h^{(l)}$).

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $h^{(l-1)}$ (for $\nabla W^{(l)} = \delta^{(l)}h^{(l-1)\top}$) and $z^{(l)}$ or $h^{(l)}$ (for $\phi'$; e.g. $\sigma' = h(1-h)$, $\tanh' = 1-h^2$, ReLU needs only the sign mask).
2. Inference can discard each activation as soon as the next layer is computed ($O(1)$ layers in memory); training must keep all $L$ layers' activations for the whole batch until the backward pass reaches them.
3. Keep only checkpoints and **recompute** the activations of a segment from its checkpoint during the backward pass: about one extra forward pass of compute
   in exchange for memory $O(L/k + k)$, minimised at $k=\sqrt L$, giving $O(\sqrt L)$.

</details>

## Part B · By hand

### Exercise 4 · Chain rule on a tiny graph
*✍️ By hand · ★☆☆*

Let $u = xy + z$ and $f = u^2$. At $x=2, y=-1, z=3$ compute $f$ and $\partial f/\partial x, \partial f/\partial y, \partial f/\partial z$
by a forward pass followed by a backward pass on the graph.

In [ ]:
f_val = None
grads = None   # [df/dx, df/dy, df/dz]

_v = np.array([2., -1, 3])
_f = lambda: (_v[0] * _v[1] + _v[2]) ** 2
check('f', f_val, _f())
check('gradient', grads, _num_grad(_f, _v), tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

Forward: $u = 1$, $f = 1$. Backward: $\bar u = \partial f/\partial u = 2u$; then a multiply gate sends $\bar u\cdot y$ to $x$ and $\bar u \cdot x$ to $y$; an add gate copies $\bar u$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Forward: $u = 2\cdot(-1)+3 = 1$, $f = 1$. Backward: $\bar u = 2u = 2$; $\partial f/\partial x = \bar u\,y = -2$, $\partial f/\partial y = \bar u\,x = 4$, $\partial f/\partial z = \bar u = 2$.
Rules of thumb: **add** gates distribute the upstream gradient unchanged, **multiply** gates swap the inputs.

```python
f_val = 1
grads = [-2, 4, 2]
```

</details>

### Exercise 5 · One logistic neuron
*✍️ By hand · ★☆☆*

For $p = \sigma(w^\top x + b)$ with BCE loss $L = -[y\log p + (1-y)\log(1-p)]$:

1. Show $\partial L/\partial z = p - y$ where $z = w^\top x + b$.
2. Compute $\nabla_w L$ and $\partial L/\partial b$ for $w = (0.5, -1)$, $b=0$, $x = (2, 1)$, $y = 1$.

In [ ]:
grad_w = None
grad_b = None

_wb = np.array([0.5, -1, 0.]); _x = np.array([2., 1, 1])
_L = lambda: -np.log(expit(_wb @ _x))
_g = _num_grad(_L, _wb)
check('dL/dw', grad_w, _g[:2], tol=1e-5)
check('dL/db', grad_b, _g[2], tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

$\sigma'(z) = \sigma(1-\sigma)$; $\partial L/\partial p = -y/p + (1-y)/(1-p)$. Multiply and simplify.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\frac{\partial L}{\partial z} = \left(-\frac yp + \frac{1-y}{1-p}\right)p(1-p) = -y(1-p) + (1-y)p = p - y$.
2. $z = 0.5\cdot2 - 1\cdot1 + 0 = 0$, $p = 0.5$, $\delta = p-y = -0.5$. $\nabla_w L = \delta x = (-1, -0.5)$, $\partial L/\partial b = -0.5$.

```python
grad_w = [-1, -0.5]
grad_b = -0.5
```

</details>

### Exercise 6 · Softmax + cross-entropy: δ = p̂ − y
*✍️ By hand · ★★☆*

1. Derive $\partial L/\partial z_j = \hat p_j - y_j$ for $L = -\sum_k y_k\log\hat p_k$, $\hat p = \mathrm{softmax}(z)$, $y$ one-hot.
2. Compute $\delta^{(L)}$ for logits $z = (1, 2, 3)$ when the true class is the **second** one.

In [ ]:
delta_hand = None   # 3 numbers

_z = np.array([1., 2, 3])
check('delta', delta_hand, _num_grad(lambda: -_sp_log_softmax(_z)[1], _z), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Use $\log\hat p_k = z_k - \log\sum_j e^{z_j}$, so $\partial\log\hat p_k/\partial z_j = [k=j] - \hat p_j$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $L = -\sum_k y_k\big(z_k - \log\sum_i e^{z_i}\big)$, so $\frac{\partial L}{\partial z_j} = -y_j + \big(\sum_k y_k\big)\hat p_j = \hat p_j - y_j$ since $\sum_k y_k=1$.
   The softmax Jacobian $\mathrm{diag}(\hat p) - \hat p\hat p^\top$ is never formed explicitly.
2. $\hat p = (0.090, 0.245, 0.665)$, $y = (0,1,0)$: $\delta = (0.090, -0.755, 0.665)$. It sums to zero: pushing up the correct logit means pushing down the others.

```python
delta_hand = [0.090, -0.755, 0.665]
```

</details>

### Exercise 7 · Shapes in a mini-batch
*✍️ By hand · ★☆☆*

A layer maps 100 → 50 features, batch size $N=32$, row convention: $H\in\mathbb R^{32\times100}$, $W\in\mathbb R^{100\times50}$, $Z = HW + b$.
Given the upstream gradient $\Delta = \partial L/\partial Z$, write the formulas and shapes of $\partial L/\partial W$, $\partial L/\partial b$ and $\partial L/\partial H$.

In [ ]:
shape_dW = None   # tuple
shape_db = None
shape_dH = None

_H = np.zeros((32, 100)); _W = np.zeros((100, 50)); _D = np.zeros((32, 50))
check('dW shape', shape_dW, (_H.T @ _D).shape)
check('db shape', shape_db, _D.sum(0).shape)
check('dH shape', shape_dH, (_D @ _W.T).shape)

<details>
<summary><b>💡 Hint</b></summary>

A gradient always has the same shape as the variable it is with respect to. Find the matrix product that produces that shape.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\partial L/\partial W = H^\top\Delta$: $(100\times32)(32\times50) = 100\times50$, the sum over the batch of $h_n\delta_n^\top$ (the note's $\delta h^\top$ is the transposed column convention).
$\partial L/\partial b = \sum_n \delta_n$: shape $(50,)$. $\partial L/\partial H = \Delta W^\top$: $32\times100$, which becomes the upstream gradient of the previous layer.

```python
shape_dW = (100, 50)
shape_db = (50,)
shape_dH = (32, 100)
```

</details>

### Exercise 8 · Full backward pass by hand
*✍️ By hand · ★★☆*

Use the network from the [[Neural Networks]] exercises: $W^{(1)} = \begin{pmatrix}1&-1\\2&0\end{pmatrix}$, $b^{(1)} = (0,-1)^\top$, ReLU,
$W^{(2)} = (3\ \ {-2})$, $b^{(2)} = 0.5$, input $x = (1,2)^\top$, target $y = 0$, loss $L = \tfrac12(\hat y - y)^2$.
The forward pass gave $z^{(1)} = (-1,1)$, $h^{(1)} = (0,1)$, $\hat y = -1.5$.
Compute $\delta^{(2)}$, $\nabla_{W^{(2)}}L$, $\delta^{(1)}$, $\nabla_{W^{(1)}}L$ and $\nabla_{b^{(1)}}L$ (math convention: $\nabla_W L = \delta h^\top$).

In [ ]:
delta2 = None
dW2 = None   # 1x2 or length 2
delta1 = None
dW1 = None   # 2x2
db1 = None

_W1 = np.array([[1., -1], [2, 0]]); _b1 = np.array([0., -1]); _W2 = np.array([3., -2]); _x = np.array([1., 2])
_L = lambda: 0.5 * (_W2 @ np.maximum(0, _W1 @ _x + _b1) + 0.5) ** 2
check('dW2', None if dW2 is None else np.ravel(dW2), _num_grad(_L, _W2), tol=1e-5)
check('dW1', dW1, _num_grad(_L, _W1), tol=1e-5)
check('db1', db1, _num_grad(_L, _b1), tol=1e-5)

<details>
<summary><b>💡 Hint 1</b></summary>

$\delta^{(2)} = \hat y - y$ (linear output + squared error). $\nabla_{W^{(2)}} = \delta^{(2)}h^{(1)\top}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\delta^{(1)} = W^{(2)\top}\delta^{(2)}\odot \mathbb 1[z^{(1)}>0]$. The inactive unit blocks the gradient.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\delta^{(2)} = -1.5$; $\nabla_{W^{(2)}} = -1.5\cdot(0, 1) = (0, -1.5)$; $\nabla_{b^{(2)}} = -1.5$.
$W^{(2)\top}\delta^{(2)} = (-4.5, 3)$, masked by ReLU' $=(0,1)$: $\delta^{(1)} = (0, 3)$.
$\nabla_{W^{(1)}} = \delta^{(1)}x^\top = \begin{pmatrix}0&0\\3&6\end{pmatrix}$, $\nabla_{b^{(1)}} = (0, 3)$.
The first hidden unit gets no gradient at all for this input: if that happens for *every* input it is a "dead ReLU" ([[Activation Functions]]).

```python
delta2 = -1.5
dW2 = [0, -1.5]
delta1 = [0, 3]
dW1 = [[0, 0], [3, 6]]
db1 = [0, 3]
```

</details>

### Exercise 9 · How fast do gradients vanish?
*✍️ By hand · ★★☆*

Consider a scalar chain $h_l = \sigma(w\,h_{l-1})$ with 10 layers, and assume every pre-activation is $0$ (so $\sigma' = 0.25$).
Then $\partial h_{10}/\partial h_0 = \prod_{l}(w\,\sigma'(z_l))$.

1. Compute this factor for $w = 1$.
2. Compute it for $w = 8$.
3. Which $w$ keeps it exactly 1, and why is that not a real fix?

In [ ]:
factor_w1 = None
factor_w8 = None
w_stable = None

_chain = lambda w: np.prod([w * expit(0.0) * (1 - expit(0.0))] * 10)
check('w = 1', factor_w1, _chain(1.0), tol=1e-12)
check('w = 8', factor_w8, _chain(8.0))
check('stable w', w_stable, 1 / (expit(0.0) * (1 - expit(0.0))))

<details>
<summary><b>💡 Hint</b></summary>

$(w\cdot 0.25)^{10}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $0.25^{10} = 9.54\times10^{-7}$: the first layer learns about a million times slower than the last.
2. $(8\cdot0.25)^{10} = 2^{10} = 1024$: explosion.
3. $w = 4$ gives 1, but only at $z=0$; with large $w$ the pre-activations quickly leave 0, $\sigma'$ drops toward 0 (saturation), and the gradient vanishes anyway.
   The real fixes are activations with $\phi'=1$ (ReLU), residual paths and normalisation.

```python
factor_w1 = 0.25 ** 10
factor_w8 = 2.0 ** 10
w_stable = 4
```

</details>

## Part C · Code: backprop with gradient checks

### Exercise 10 · A numerical gradient and a relative error
*💻 Code · ★☆☆*

Implement `num_grad(f, x, eps=1e-5)`: the central-difference gradient $\frac{f(x+\varepsilon e_i) - f(x-\varepsilon e_i)}{2\varepsilon}$ of a
scalar function `f(x)` for an array `x` of **any shape**, and `rel_error(a, b)` $= \max_i\frac{|a_i-b_i|}{\max(|a_i|+|b_i|, 10^{-12})}$.
Test on $f(x) = \sum_i x_i^2\sin x_i$ whose gradient is $2x\sin x + x^2\cos x$.

In [ ]:
def num_grad(f, x, eps=1e-5):
    # TODO: loop over all entries (np.nditer or x.flat), perturb, evaluate, restore
    return None

def rel_error(a, b):
    return None

x_t = rng.normal(size=(3, 4))
f_t = lambda x: np.sum(x ** 2 * np.sin(x))
g_num = num_grad(f_t, x_t.copy())

_g_true = 2 * x_t * np.sin(x_t) + x_t ** 2 * np.cos(x_t)
check('num_grad matches analytic gradient', g_num, _g_true, tol=1e-7)
check('rel_error small', None if g_num is None or rel_error(g_num, _g_true) is None else rel_error(g_num, _g_true) < 1e-8, True)

<details>
<summary><b>💡 Hint</b></summary>

Work on a flat view: `xf = x.reshape(-1)` shares memory with `x` (for a contiguous array), so perturbing `xf[i]` perturbs `x`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The central difference has error $O(\varepsilon^2)$ (the one-sided version only $O(\varepsilon)$); too small an $\varepsilon$ makes floating-point
cancellation dominate. Rule of thumb for float64: relative error $<10^{-7}$ is fine, $>10^{-3}$ is almost certainly a bug.

```python
def num_grad(f, x, eps=1e-5):
    x = np.asarray(x, dtype=float)
    g = np.zeros_like(x)
    xf, gf = x.reshape(-1), g.reshape(-1)
    for i in range(xf.size):
        old = xf[i]
        xf[i] = old + eps; fp = f(x)
        xf[i] = old - eps; fm = f(x)
        xf[i] = old
        gf[i] = (fp - fm) / (2 * eps)
    return g

def rel_error(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return np.max(np.abs(a - b) / np.maximum(np.abs(a) + np.abs(b), 1e-12))

x_t = rng.normal(size=(3, 4))
f_t = lambda x: np.sum(x ** 2 * np.sin(x))
g_num = num_grad(f_t, x_t.copy())
```

</details>

### Exercise 11 · A Linear layer with forward and backward
*💻 Code · ★★☆*

Implement a `Linear` layer: `forward(X)` returns `X @ W + b` and caches `X`; `backward(dZ)` stores `self.dW`, `self.db` and returns `dX`.
The check uses the trick "loss $= \sum Z\odot G$ for a random $G$", whose upstream gradient is exactly $G$.

In [ ]:
class Linear:
    def __init__(self, n_in, n_out, rng):
        self.W = rng.normal(0, 1 / np.sqrt(n_in), (n_in, n_out))
        self.b = np.zeros(n_out)
    def forward(self, X):
        return None   # TODO
    def backward(self, dZ):
        self.dW = None   # TODO
        self.db = None
        return None

lin = Linear(4, 3, np.random.default_rng(1))
X_l = rng.normal(size=(5, 4)); G_l = rng.normal(size=(5, 3))

_out = lin.forward(X_l)
if _out is None:
    check('dW', None, 0); check('db', None, 0); check('dX', None, 0)
else:
    _dX = lin.backward(G_l)
    _loss = lambda: np.sum(lin.forward(X_l) * G_l)
    check('dW', lin.dW, _num_grad(_loss, lin.W))
    check('db', lin.db, _num_grad(_loss, lin.b))
    check('dX', _dX, _num_grad(_loss, X_l))

<details>
<summary><b>💡 Hint</b></summary>

$dW = X^\top dZ$, $db = \sum_n dZ_n$ (sum over rows), $dX = dZ\,W^\top$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

These three lines are the whole of backprop for affine layers; every framework's `Linear.backward` looks like this.

```python
class Linear:
    def __init__(self, n_in, n_out, rng):
        self.W = rng.normal(0, 1 / np.sqrt(n_in), (n_in, n_out))
        self.b = np.zeros(n_out)
    def forward(self, X):
        self.X = X
        return X @ self.W + self.b
    def backward(self, dZ):
        self.dW = self.X.T @ dZ
        self.db = dZ.sum(axis=0)
        return dZ @ self.W.T

lin = Linear(4, 3, np.random.default_rng(1))
X_l = rng.normal(size=(5, 4)); G_l = rng.normal(size=(5, 3))
```

</details>

### Exercise 12 · Activation layers
*💻 Code · ★☆☆*

Implement `Tanh`, `Sigmoid` and `ReLU` layers with `forward(Z)` (cache what you need) and `backward(dH)` returning `dZ`.
Use the cached **output** where possible ($\sigma' = h(1-h)$, $\tanh' = 1-h^2$).

In [ ]:
class Tanh:
    def forward(self, Z): return None
    def backward(self, dH): return None

class Sigmoid:
    def forward(self, Z): return None
    def backward(self, dH): return None

class ReLU:
    def forward(self, Z): return None
    def backward(self, dH): return None

Z_a = rng.normal(size=(4, 5)); G_a = rng.normal(size=(4, 5))

for _cls in (Tanh, Sigmoid, ReLU):
    _layer = _cls()
    _out = _layer.forward(Z_a)
    if _out is None:
        check(_cls.__name__, None, 0); continue
    _dZ = _layer.backward(G_a)
    check(_cls.__name__ + ' backward', _dZ, _num_grad(lambda: np.sum(_cls().forward(Z_a) * G_a), Z_a))

<details>
<summary><b>💡 Hint</b></summary>

ReLU: cache the mask `Z > 0` and return `dH * mask`. The random `Z_a` is almost surely never exactly 0, so the kink does not matter here.

</details>

<details>
<summary><b>✅ Solution</b></summary>

An activation layer's backward is an element-wise product with $\phi'(z)$, the $\odot\phi'(z^{(l)})$ in the note's recursion.

```python
class Tanh:
    def forward(self, Z):
        self.h = np.tanh(Z); return self.h
    def backward(self, dH):
        return dH * (1 - self.h ** 2)

class Sigmoid:
    def forward(self, Z):
        self.h = expit(Z); return self.h
    def backward(self, dH):
        return dH * self.h * (1 - self.h)

class ReLU:
    def forward(self, Z):
        self.mask = Z > 0; return Z * self.mask
    def backward(self, dH):
        return dH * self.mask

Z_a = rng.normal(size=(4, 5)); G_a = rng.normal(size=(4, 5))
```

</details>

### Exercise 13 · Backprop through a full MLP (gradient-checked)
*💻 Code · ★★☆*

Write `loss_and_grads(params, X, y)` for a 2-hidden-layer tanh MLP with softmax output and mean cross-entropy.
`params = [W1, b1, W2, b2, W3, b3]`; return `(loss, grads)` with `grads` in the same order. Implement the backward pass by hand
(you may reuse your layers). It must pass a gradient check with relative error $<10^{-6}$.

In [ ]:
def loss_and_grads(params, X, y):
    # TODO
    return None, None

_r = np.random.default_rng(2)
sizes = [5, 7, 6, 3]
params = []
for a, b in zip(sizes[:-1], sizes[1:]):
    params += [_r.normal(0, 1 / np.sqrt(a), (a, b)), 0.1 * _r.normal(size=b)]
X_g = _r.normal(size=(8, 5)); y_g = _r.integers(0, 3, size=8)
loss_g, grads_g = loss_and_grads(params, X_g, y_g)

def _ref_loss():
    H = X_g
    for i in range(0, 4, 2):
        H = np.tanh(H @ params[i] + params[i + 1])
    return -_sp_log_softmax(H @ params[4] + params[5], axis=1)[np.arange(len(y_g)), y_g].mean()
check('loss', loss_g, _ref_loss())
for _i, _name in enumerate(['W1', 'b1', 'W2', 'b2', 'W3', 'b3']):
    check('grad ' + _name, None if grads_g is None else grads_g[_i], _num_grad(_ref_loss, params[_i]), tol=1e-7)

<details>
<summary><b>💡 Hint 1</b></summary>

Output: $\Delta_3 = (\mathrm{softmax}(Z_3) - Y)/N$ with one-hot $Y$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Then for $l = 3, 2, 1$: $\nabla W_l = H_{l-1}^\top\Delta_l$, $\nabla b_l = \sum\Delta_l$, $\Delta_{l-1} = (\Delta_l W_l^\top)\odot(1 - H_{l-1}^2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

This is the note's recursion in row convention. The $1/N$ from the mean is applied once, at the top; every gradient below inherits it.
Tanh (smooth) is chosen so the finite-difference check is not disturbed by ReLU kinks.

```python
def loss_and_grads(params, X, y):
    W1, b1, W2, b2, W3, b3 = params
    N = len(y)
    H1 = np.tanh(X @ W1 + b1)
    H2 = np.tanh(H1 @ W2 + b2)
    Z3 = H2 @ W3 + b3
    loss = -_sp_log_softmax(Z3, axis=1)[np.arange(N), y].mean()
    D3 = _sp_softmax(Z3, axis=1); D3[np.arange(N), y] -= 1; D3 /= N
    gW3, gb3 = H2.T @ D3, D3.sum(0)
    D2 = (D3 @ W3.T) * (1 - H2 ** 2)
    gW2, gb2 = H1.T @ D2, D2.sum(0)
    D1 = (D2 @ W2.T) * (1 - H1 ** 2)
    gW1, gb1 = X.T @ D1, D1.sum(0)
    return loss, [gW1, gb1, gW2, gb2, gW3, gb3]

_r = np.random.default_rng(2)
sizes = [5, 7, 6, 3]
params = []
for a, b in zip(sizes[:-1], sizes[1:]):
    params += [_r.normal(0, 1 / np.sqrt(a), (a, b)), 0.1 * _r.normal(size=b)]
X_g = _r.normal(size=(8, 5)); y_g = _r.integers(0, 3, size=8)
loss_g, grads_g = loss_and_grads(params, X_g, y_g)
```

</details>

### Exercise 14 · Micrograd: reverse-mode autodiff for scalars
*💻 Code · ★★★*

Following Karpathy's *micrograd*, complete a `Value` class supporting `+`, `*`, `tanh()` and `backward()`.
Each operation creates a new node that remembers its parents and a `_backward` closure that adds the local
gradient times `out.grad` to the parents' `.grad`. `backward()` topologically sorts the graph and calls `_backward` in reverse order.
Test it on Exercise 4 ($f = (xy+z)^2$, written as `u*u`) and on a neuron $\tanh(w_1x_1 + w_2x_2 + b)$.

In [ ]:
class Value:
    def __init__(self, data, _parents=()):
        self.data = float(data)
        self.grad = 0.0
        self._parents = _parents
        self._backward = lambda: None
    def __add__(self, other):
        raise NotImplementedError   # TODO
    def __mul__(self, other):
        raise NotImplementedError   # TODO
    def tanh(self):
        raise NotImplementedError   # TODO
    def backward(self):
        raise NotImplementedError   # TODO

try:
    x, y, z = Value(2.0), Value(-1.0), Value(3.0)
    u = x * y + z
    f = u * u
    f.backward()
    check('micrograd on f = (xy+z)^2', [x.grad, y.grad, z.grad], [-2, 4, 2])
    _p = np.array([0.3, -0.7, 0.1, 1.5, 0.4])   # w1, w2, b, x1, x2
    w1, w2, b, x1, x2 = [Value(v) for v in _p]
    o = (w1 * x1 + w2 * x2 + b).tanh()
    o.backward()
    _ref = _num_grad(lambda: np.tanh(_p[0] * _p[3] + _p[1] * _p[4] + _p[2]), _p)
    check('micrograd neuron', [w1.grad, w2.grad, b.grad, x1.grad, x2.grad], _ref, tol=1e-6)
except NotImplementedError:
    check('micrograd on f = (xy+z)^2', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

For `out = self * other`: `self.grad += other.data * out.grad` and `other.grad += self.data * out.grad`. Use `+=` because a node can be used several times (like `u*u`).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

backward: build `topo` with a DFS over `_parents` (visited set), set `self.grad = 1.0`, then `for v in reversed(topo): v._backward()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Accumulating with `+=` implements the multivariate chain rule: a node used in several places receives the **sum** of the gradients from each use
(here $u$ is used twice in $u\cdot u$, giving $2u$). The topological order guarantees a node's `.grad` is complete before it is propagated further.
This is exactly what PyTorch's autograd does with tensors instead of scalars.

```python
class Value:
    def __init__(self, data, _parents=()):
        self.data = float(data)
        self.grad = 0.0
        self._parents = _parents
        self._backward = lambda: None
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    def tanh(self):
        t = np.tanh(self.data)
        out = Value(t, (self,))
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out
    def backward(self):
        topo, seen = [], set()
        def build(v):
            if v not in seen:
                seen.add(v)
                for p in v._parents:
                    build(p)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()
```

</details>

### Exercise 15 · Measure vanishing gradients
*💻 Code · ★★☆*

Build a 20-layer MLP (width 64, input batch of 128 Gaussian samples, loss = mean of the last layer's outputs squared / 2) twice:
(a) sigmoid activations with Xavier init $W\sim\mathcal N(0, 1/n_{in})$; (b) ReLU with He init $W\sim\mathcal N(0, 2/n_{in})$.
Backpropagate (by hand, with your layers) and store the Frobenius norms $\lVert\nabla_{W^{(l)}}L\rVert$ for $l = 1..20$ in arrays `gn_sigmoid` and `gn_relu`.
Compare the ratio first-layer / last-layer norm for both.

In [ ]:
def grad_norms(act='sigmoid', depth=20, width=64, seed=0):
    # TODO: forward through `depth` layers caching inputs, then backward; return array of ||dW_l||
    return None

gn_sigmoid = grad_norms('sigmoid')
gn_relu = grad_norms('relu')

if gn_sigmoid is None or gn_relu is None:
    check('sigmoid vanishes, ReLU+He does not', None, True)
else:
    print('ratio first/last  sigmoid: %.2e   relu: %.2e' % (gn_sigmoid[0] / gn_sigmoid[-1], gn_relu[0] / gn_relu[-1]))
    check('sigmoid vanishes, ReLU+He does not', gn_sigmoid[0] / gn_sigmoid[-1] < 1e-4 and 0.05 < gn_relu[0] / gn_relu[-1] < 20, True)

<details>
<summary><b>💡 Hint</b></summary>

Keep lists `Hs` (layer inputs) and `Zs`. Upstream at the top: $\Delta = H_L/N$. Then `dW = H_{l-1}.T @ (Δ ⊙ φ'(Z_l))`, `Δ ← (Δ ⊙ φ'(Z_l)) @ W_l.T`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With sigmoid the gradient norm drops by roughly a factor $\approx 4$–$5$ per layer (the $0.25$ from $\sigma'$ times $\lVert W\rVert\approx 1$), so after 20 layers the
first layer's gradient is many orders of magnitude smaller than the last one's. With ReLU + He init, the variance is preserved forward and
backward, so the norms stay within a small factor across depth. This is why ReLU + He was a breakthrough for training deep nets ([[Activation Functions]], [[Training Tricks]]).

```python
def grad_norms(act='sigmoid', depth=20, width=64, seed=0):
    r = np.random.default_rng(seed)
    N = 128
    H = r.normal(size=(N, width))
    gain = 1.0 if act == 'sigmoid' else 2.0
    Ws = [r.normal(0, np.sqrt(gain / width), (width, width)) for _ in range(depth)]
    Hs, Zs = [], []
    for W in Ws:
        Hs.append(H)
        Z = H @ W
        Zs.append(Z)
        H = expit(Z) if act == 'sigmoid' else np.maximum(0, Z)
    D = H / N   # dL/dH for L = sum(H**2) / (2N)
    norms = np.zeros(depth)
    for l in reversed(range(depth)):
        dphi = expit(Zs[l]) * (1 - expit(Zs[l])) if act == 'sigmoid' else (Zs[l] > 0)
        Dz = D * dphi
        norms[l] = np.linalg.norm(Hs[l].T @ Dz)
        D = Dz @ Ws[l].T
    return norms

gn_sigmoid = grad_norms('sigmoid')
gn_relu = grad_norms('relu')
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Backpropagation](Backpropagation.md).*